# Piping a RunnableParallel with Other Runnables

In [1]:
%load_ext dotenv
%dotenv

In [2]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

In [3]:
chat_template_books = ChatPromptTemplate.from_template(
    '''
    Suggest three of the best intermediate-level {programming language} books. 
    Answer only by listing the books.
    '''
)

chat_template_projects = ChatPromptTemplate.from_template(
    '''
    Suggest three interesting {programming language} projects suitable for intermediate-level programmers. 
    Answer only by listing the projects.
    '''
)

chat_template_time = ChatPromptTemplate.from_template(
     '''
     I'm an intermediate level programmer.
     
     Consider the following literature:
     {books}
     
     Also, consider the following projects:
     {projects}
     
     Roughly how much time would it take me to complete the literature and the projects?
     
     '''
)

In [4]:
chat = ChatOpenAI(model_name = 'gpt-4', 
                  seed = 365,
                  temperature = 0,
                  max_tokens = 500)

In [5]:
string_parser = StrOutputParser()

In [6]:
chain_books = chat_template_books | chat | string_parser

chain_projects = chat_template_projects | chat | string_parser

In [7]:
chain_parallel = RunnableParallel({'books':chain_books, 'projects':chain_projects})

In [8]:
chain_parallel.invoke({'programming language':'Python'})

{'books': '1. "Fluent Python: Clear, Concise, and Effective Programming" by Luciano Ramalho\n2. "Python Cookbook: Recipes for Mastering Python 3" by David Beazley and Brian K. Jones\n3. "Effective Python: 90 Specific Ways to Write Better Python" by Brett Slatkin',
 'projects': '1. Building a Weather Forecasting Application using APIs\n2. Developing a Web Scraper for E-commerce Websites\n3. Creating a Text-based Adventure Game with Multiple Decision Paths'}

In [9]:
chain_time1 = (RunnableParallel({'books':chain_books, 
                                'projects':chain_projects}) 
              | chat_template_time 
              | chat 
              | string_parser
             )

In [10]:
chain_time2 = ({'books':chain_books, 
                'projects':chain_projects}
              | chat_template_time 
              | chat 
              | string_parser
             )

In [11]:
print(chain_time2.invoke({'programming language':'Python'}))

The time it would take to complete the literature and the projects can vary greatly depending on several factors such as your current skill level, the amount of time you can dedicate each day, your reading speed, and how quickly you grasp new concepts.

For the literature:

1. "Fluent Python: Clear, Concise, and Effective Programming" - This book is around 800 pages. If you read and practice for about 2 hours a day, it might take you around 4-6 weeks to complete.

2. "Python Cookbook: Recipes for Mastering Python 3" - This book is approximately 700 pages. Again, assuming 2 hours of reading and practicing per day, it might take you around 3-5 weeks.

3. "Effective Python: 90 Specific Ways to Write Better Python" - This book is shorter, around 230 pages. At the same pace, you might finish it in 1-2 weeks.

For the projects:

1. Building a Web Scraper using BeautifulSoup - If you're already familiar with the basics of web scraping, you could potentially build a simple web scraper in a few

In [12]:
chain_time2.get_graph().print_ascii()

            +-------------------------------+              
            | Parallel<books,projects>Input |              
            +-------------------------------+              
                   ***               ***                   
                ***                     ***                
              **                           **              
+--------------------+              +--------------------+ 
| ChatPromptTemplate |              | ChatPromptTemplate | 
+--------------------+              +--------------------+ 
           *                                   *           
           *                                   *           
           *                                   *           
    +------------+                      +------------+     
    | ChatOpenAI |                      | ChatOpenAI |     
    +------------+                      +------------+     
           *                                   *           
           *                            